# fMRI 预处理与质量控制：先看懂完整流程

fMRI 分析的难点不只在预处理。采集条件、质量控制、去噪策略、分析选择和结果解释都会影响结论。本 Notebook 先介绍从原始数据到可分析影像的流程，再用项目提供的一个 ADHD-200 样例练习检查影像和混杂变量。

> **重要边界：**本项目的样例已经过空间预处理。下面不会从原始扫描运行 slice-timing、运动校正、畸变校正或标准化，也不会替代 fMRIPrep 的 HTML 质量报告。

## 一张路线图：原始数据如何变成可分析数据？

典型流程可以概括为：

1. **采集与整理：**确认扫描协议、相位编码/场图等校正信息，并按 BIDS 组织影像和元数据。
2. **预处理：**视数据和研究问题进行切片时间校正、头动校正、磁敏感畸变校正、配准到结构像、标准空间变换；平滑是可选步骤，不是固定必选项。
3. **质量控制：**检查头动、配准、脑覆盖、异常时间点及处理报告；发现失败时先调查原因，而不是继续分析。
4. **分析准备：**选择脑区或体素表示，回归混杂变量，按问题进行去趋势和滤波。
5. **统计与解释：**例如本项目后续计算单个受试者的 ROI 功能连接；相关性不是因果关系。

预处理不是越多越好：不合适的校正或重复处理也会扭曲数据。像 fMRIPrep 这样的流水线能执行标准步骤并生成报告，但研究者仍需理解输入、输出和质量控制。

## 1. 读取一份样例影像

NIfTI header 能告诉我们空间维度、体素大小、时间采样间隔（TR）和空间变换等信息。先记录这些可核对的事实；不要只凭文件名推断空间方向或采集参数。首次运行需要联网下载样例，之后 Nilearn 会使用本地缓存。

In [ ]:
from nilearn import datasets, image

dataset = datasets.fetch_adhd(n_subjects=1)
func_file = dataset.func[0]
func_img = image.load_img(func_file)

print(f"文件：{func_file}")
print(f"维度 (x, y, z, time)：{func_img.shape}")
print(f"体素尺寸与 TR：{func_img.header.get_zooms()}")
print(f"空间/时间单位：{func_img.header.get_xyzt_units()}")
print(f"空间仿射矩阵：\n{func_img.affine}")
assert len(func_img.shape) == 4, f"预期 4D 功能像，实际形状为 {func_img.shape}"

**暂停想一想：**第四维代表什么？当前图像有多少个时间点？header 中的 TR 单位是什么？这些信息可帮助发现格式或元数据问题，但单靠它们无法判断头动是否可接受。

## 2. 看一个时间点的脑图像

下面显示时间序列中间的 volume。它适合初步确认图像是否可读、脑部是否大致在视野内；单张图不能证明配准正确，也不能替代多时间点检查和正式 QC 报告。

In [ ]:
from nilearn import plotting

middle_volume = func_img.shape[3] // 2
middle_img = image.index_img(func_img, middle_volume)
plotting.plot_img(
    middle_img,
    display_mode="ortho",
    title=f"Middle volume: {middle_volume}",
)

## 3. 看看混杂变量是什么

头动参数描述每个时间点的位移/旋转；白质和脑脊液信号常作为额外混杂变量候选。它们不是神经活动本身。检查表格列名、行数和缺失值，有助于确认 confounds 与影像时间点是否对应。具体选哪些变量应由研究方案决定。

In [ ]:
import pandas as pd

confounds_file = dataset.confounds[0]
confounds = pd.read_csv(confounds_file, sep=None, engine="python")
print(f"混杂变量文件：{confounds_file}")
print(f"表格形状：{confounds.shape}（行 × 列）")
print(f"影像时间点：{func_img.shape[3]}")
print(f"缺失值最多的列：\n{confounds.isna().sum().sort_values(ascending=False).head()}")
confounds.head()

**质量控制思考：**表格行数是否与影像时间点数量一致？哪些列看起来像运动参数？若有高头动或异常 volume，应进一步查看运动曲线和报告，并依照方案决定如何处理。不要仅凭本表或这张示例图宣称样例通过了完整 QC。

后续的功能连接 Notebook 会展示混杂回归、去趋势、滤波和 ROI 提取。这些属于分析前的时间序列清理，不等同于完整的空间预处理。